In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir transformers bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-d9zn00bt/unsloth_3f3e40cefc51404f87e469e0ae1bd2ef
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-d9zn00bt/unsloth_3f3e40cefc51404f87e469e0ae1bd2ef
  Resolved https://github.com/unslothai/unsloth.git to commit cd65584f1972e3e5a021665b21c8eb391a677acd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 320.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 431.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 277.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 389.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 211.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 399.8 MB/s eta 0:00:

In [ ]:
PROMPT_TEMPLATE = """
You are an AI assistant that evaluates a student's concept flow graph for an educational exercise.

IMPORTANT: The backend has already computed which topics are matched and missing.
You MUST use these counts exactly and MUST NOT invent additional missing topics.

==============================
COURSE CONTEXT
==============================
{context}

==============================
USER FLOW TOPICS (extracted)
==============================
{flow_topics}

==============================
EXPECTED TOPICS FOUND IN USER FLOW
==============================
{matching_topics}

==============================
EXPECTED TOPICS MISSING FROM USER FLOW
==============================
{missing_topics}

==============================
FLOW GRAPH JSON
==============================
{flow_json}

==============================
SCORING FACTS (PRECOMPUTED)
==============================
Total expected topics: {total_db}
Missing MAIN topics: {missing_main_count}

==============================
SCORING GUIDELINES
==============================

Base score = 1.0
- Missing MAIN topic penalty = up to 1.0 total
missing_main_penalty = min(1.0, missing_main_count * (1.0 / total_expected_topics))
score = max(0.0, 1.0 - missing_main_penalty)
Round the score to two decimals.

EXAMPLE SCORES:
- If missing_main_count = 0 → score = 1.0 (perfect)
- If missing_main_count = 1 and total_expected_topics = 12 → penalty = 0.08 → score = 0.92
- If missing_main_count = 2 and total_expected_topics = 12 → penalty = 0.17 → score = 0.83
- If missing_main_count = 12 and total_expected_topics = 12 → penalty = 1.0 → score = 0.0

==============================
VALIDATION TASK
==============================

Evaluate the student's flow graph based on the following checks:

1. Topic Relevance
   - Determine whether the topics included in the student's graph are relevant to the course context.
   - Check whether the topic descriptions reflect the core meaning of the concepts from the course material.

2. Graph Connection Validation
   Examine the edges in FLOW GRAPH JSON. Each connection is represented as:

   {{
     "source": "Topic A",
     "target": "Topic B"
   }}

   Example edges:

   {{
     "source": "Deep Learning",
     "target": "Neural Networks"
   }}
   {{
     "source": "Machine Learning",
     "target": "Supervised Learning"
   }}
   {{
     "source": "Machine Learning",
     "target": "Unsupervised Learning"
   }}
   {{
     "source": "Machine Learning",
     "target": "Reinforcement Learning"
   }}
   {{
     "source": "Deep Learning",
     "target": "Computer Vision"
   }}
   {{
     "source": "Deep Learning",
     "target": "Natural Language Processing"
   }}

   2.1 Logical Relationships
   - Verify whether the source and target topics have a valid conceptual relationship based on the course context.

   2.2 Direction of Relationships
   - Confirm the edge direction is logical: broader concepts as source, specific concepts as target.

   2.3 Hierarchical Structure
   - Ensure edges respect the conceptual hierarchy.

   2.4 Cross-Domain Relationships
   - Check whether edges between different domains are meaningful.

   2.5 Incorrect or Illogical Connections
   - Identify edges not supported by the course context.

3. Missing Topic Identification
   - Identify missing topics using ONLY the "EXPECTED TOPICS MISSING FROM USER FLOW" list.

4. Structural Quality of the Graph
   - Evaluate whether the overall graph structure reflects the conceptual relationships in the course context.

   IMPORTANT SCORING NOTE:
   - First, calculate the base score from missing topics only
   - Then, you may adjust the score DOWN by up to 0.15 based on structural quality issues
   - Do NOT adjust the score up - only down if there are issues
   - The final score must still be between 0.0 and 1.0

Provide a clear explanation in the feedback field.

==============================
OUTPUT FORMAT
==============================

Return ONLY valid JSON:

{{
  "feedback": "<clear explanation of strengths, missing topics, and structural issues>",
  "points": <score between 0.0 and 1.0 rounded to two decimals>
}}

DO NOT include markdown.
DO NOT include extra text.
DO NOT recompute missing topics.
"""

In [ ]:
def _fmt_topic_list(topics: list[str]) -> str:
    """Format a list of topic names for prompt display."""
    if not topics:
        return "(none)"
    return "\n".join(f"  - {t}" for t in topics)

def format_conversation(example):
    context = example["input"]["context"]

    # Extract just topic names for _fmt_topic_list
    flow_topics = [t["topic"] for t in example["input"]["flow_topics"]]
    matching_topics = [t["topic"] for t in example["input"]["matching_topics"]]
    missing_topics = [t["topic"] for t in example["input"]["missing_topics"]]

    flow_edges = example["input"]["flow_edges"]
    missing_main_count = example["input"]["missing_main_count"]
    total_db = len(matching_topics) + len(missing_topics)

    flow_json = {
        "nodes": example["input"]["flow_topics"],  # keep full nodes with details
        "edges": flow_edges
    }

    user_content_prompt = PROMPT_TEMPLATE.format(
        context=context,
        flow_topics=_fmt_topic_list(flow_topics),
        matching_topics=_fmt_topic_list(matching_topics),
        missing_topics=_fmt_topic_list(missing_topics),
        flow_json=json.dumps(flow_json, indent=2),
        total_db=total_db,
        missing_main_count=missing_main_count
    )
    return user_content_prompt

In [ ]:
def parse_response(response_text):
    import re
    if not response_text:
        return {'points': 0, 'feedback': 'Error: No response'}

    assistant_patterns = [
        r'assistant\s*\n',  # "assistant" followed by newline
        r'assistant\s*\n',  # "assistant" followed by newline
        r'^assistant\s*$',   # "assistant" on its own line
        r'\nassistant\n',    # "assistant" between newlines
        r'assistant\s*\n\n', # "assistant" with blank line after
    ]

    # Try each pattern to find where assistant's response starts
    for pattern in assistant_patterns:
        match = re.search(pattern, response_text, re.MULTILINE | re.IGNORECASE)
        if match:
            # Extract everything after the assistant marker
            response_text = response_text[match.end():]
            break

    print(response_text)
    try:
        points_match = re.search(r'"points"\s*:\s*([\d.]+)', response_text, re.IGNORECASE)
        feedback_match = re.search(r'"feedback"\s*:\s*"([^"]*(?:\\"[^"]*)*)"', response_text, re.DOTALL | re.IGNORECASE)

        if points_match and feedback_match:
            return {
                'points': float(points_match.group(1)),
                'feedback': feedback_match.group(1).replace('\\"', '"')
            }
        else:
            return {'points': 0, 'feedback': 'Error: Could not extract evaluation data'}
    except Exception as e:
        return {'points': 0, 'feedback': f'Error parsing: {e}'}

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/llama-3.2-3b-instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=3084,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNo

In [ ]:
def call_llama(model, tokenizer, prompt):
  FastLanguageModel.for_inference(model)

  messages = [
      {
          "role": "system",
          "content": "You are an AI assistant that evaluates a student's concept flow graph for an educational exercise."
      },
      {
          "role": "user",
          "content": prompt
      }
  ]

  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False,
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt",
      padding=True,
      truncation=True,
  )

  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  input_len = inputs["input_ids"].shape[1]

  outputs = model.generate(
      **inputs,
      max_new_tokens=1024,
      use_cache=False,
      temperature=0.1,
      do_sample=False,
      repetition_penalty=1.1,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
      num_beams=1,          # Use greedy decoding
      early_stopping=True,
  )

  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return response

In [ ]:
import json
from google.colab import files
import os
import shutil

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_PATH = "evaluation_results_llama3.2.json"
DRIVE_PATH = f"/content/drive/MyDrive/{OUTPUT_PATH}"
DATASET_PATH = "/content/drive/MyDrive/mindmap_evaluation_dataset_200.json"

data = json.load(open(DATASET_PATH, "r"))
data = data[:100]

# Load previous progress if it exists
if os.path.exists(DRIVE_PATH):
    with open(DRIVE_PATH, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Resuming from {len(results)} completed entries...")
else:
    results = []
    print("No previous results found, starting fresh...")

start_idx = len(results)

print(f"Starting evaluation on {len(data)} entries...")

for i in range(start_idx, len(data)):
    entry = data[i]
    print(f'[{i+1}/{len(data)}] Evaluating...')

    ground_truth = entry.get('output', {})
    prompt = format_conversation(entry)

    res_llama = call_llama(model, tokenizer, prompt)
    parsed_llama = parse_response(res_llama)

    result = {
        'id': i,
        'ground_truth': {
            'points': ground_truth.get('points', 0),
            'feedback': ground_truth.get('feedback', '')
        },
        'llama3.2': parsed_llama
    }

    results.append(result)

    # Save after every entry
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)

    shutil.copy(OUTPUT_PATH, DRIVE_PATH)
    print(f"Saved progress to: {DRIVE_PATH}")

print("Llama3.2 Evaluation complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
['.config', 'huggingface_tokenizers_cache', 'unsloth_compiled_cache', 'drive', 'sample_data']
Starting evaluation on 100 entries...
[1/100] Evaluating...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.

Here is the evaluation of the student's flow graph:


{
  "feedback": "The student's flow graph shows some connections between topics but lacks depth and relevance to the course context. Specifically, the student missed three main topics: Neural Networks, Unsupervised Learning, and Robotics. Additionally, some edges do not follow logical relationships, such as 'Machine Learning' being both the source and target of multiple edges. Furthermore, the graph could benefit from more hierarchical structure and fewer cross-domain relationships. However, the student did demonstrate some understanding of the course material, particularly in their connections between 'Deep Learning', 'Machine Learning', and 'Computer Vision'. Overall, the student needs to improve their understanding of the course material and develop more logical connections between topics.",
  "points": 0.68
}
[2/100] Evaluating...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "feedback": "The student's flow graph shows some relevant topics but lacks others crucial to the course context. Specifically, the topics 'Cells', 'Respiration', and 'Enzymes' are missing. Additionally, the graph could benefit from more connections between topics, particularly between 'Biology' and 'Ecosystems'. However, the overall structure of the graph respects the conceptual hierarchy and logical relationships between topics. To improve the score, the student should focus on including all required topics and adding more connections between them.",
  "points": 0.85
}
```
[3/100] Evaluating...


...

```json
{
  "feedback": "The student's flow graph shows some relevant topics but lacks key concepts like Machine Learning, Deep Learning, and Reinforcement Learning. The connections between topics seem mostly logical, with some minor exceptions. However, the graph could benefit from more explicit connections between topics and might be improved by adding more edges. Overall, the student demonstrates a good understanding of the course context, but there is room for improvement in terms of completeness and structural quality.",
  "points": 0.68
}
```
[49/100] Evaluating...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "feedback": "The student's flow graph shows some relevant topics but lacks key concepts like Artificial Intelligence, Deep Learning, Reinforcement Learning, Natural Language Processing, and Data Science. The graph also contains some illogical connections such as 'Robotics' being connected to 'Neural Nets', which does not make sense in this context. Additionally, the graph could benefit from more connections between different domains. However, the student has correctly identified the main topics missing from the user flow. Overall, the graph structure is mostly correct but could be improved with more connections and clarity.",
  "points": 0.68
}
```
[50/100] Evaluating...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "feedback": "The student's flow graph shows some relevant topics but lacks key concepts like Supervised Learning, Unsupervised Learning, and Ethics. The connections between topics seem mostly logical, with some minor exceptions. However, the graph could benefit from more explicit connections between certain domains. Overall, the student demonstrates a good foundation in AI-related topics but needs to expand their knowledge in other areas.",
  "points": 0.85
}
```

Note: I've adjusted the score downward due to the identified missing topics and potential structural issues, while ensuring it remains within the allowed range of 0.0 to 1.0.
[51/100] Evaluating...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
